# Notebook 2 – Construction of the Augmented Dataset (Spanish)

1. Trial dataset ES (4) + translations CA→ES (7) + IT→ES (8) = **19 samples**
2. Back-translation with different language chains per round to maximise lexical diversity while preserving meaning and line break format (`\n`).

---

### Back-translation strategy by round


| Round | Chain | Why |
|---|---|---|
| **Round 1** | ES → EN → DE → EN → ES | German has very different syntactic structure → greater syntactic variation |
| **Round 2** | ES → FR → PT → ES | Romance languages with different vocabulary → moderate lexical variation |
| **Round 3** | ES → EN → ZH → EN → ES | Chinese forces complete restructuring (isolating morphology) → maximum variation |

Back-translation is always applied to the original base samples.

---

### Expected data directory structure

```
data/
└── trial/
    ├── es_trial_document.csv
    ├── it_trial_document.csv
    └── cat_trial_document.csv
```

### Requirements

A valid Groq API key is required. Set it as an environment variable before running:
```bash
export GROQ_API_KEY=your_key_here
```

## Installation

In [ ]:
!pip install groq pandas nltk matplotlib

## 1. Imports and setup

In [ ]:
import os
import time
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from groq import Groq

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise ValueError("Set the GROQ_API_KEY environment variable before running.")

MODEL = "llama-3.3-70b-versatile"
TEMP  = 0.3
DELAY = 1.2

client = Groq(api_key=GROQ_API_KEY)
print(f'Groq client initialised. Model: {MODEL}')

## 2. Load trial datasets

In [ ]:
PATH_ES  = 'data/trial/es_trial_document.csv'
PATH_CAT = 'data/trial/cat_trial_document.csv'
PATH_IT  = 'data/trial/it_trial_document.csv'

df_es  = pd.read_csv(PATH_ES)
df_cat = pd.read_csv(PATH_CAT)
df_it  = pd.read_csv(PATH_IT)

print(f'ES:  {len(df_es)} seg. | CAT: {len(df_cat)} seg. | IT: {len(df_it)} seg.')

## 3. Helper function: LLM call

In [ ]:
def call_llm(system: str, user: str,
              model: str = MODEL, temperature: float = TEMP,
              max_retries: int = 3) -> str:
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user',   'content': user},
                ]
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f'  [Attempt {attempt}/{max_retries}] Error: {e}')
            time.sleep(DELAY * attempt)
    return ''

print('call_llm defined.')

## 4. Translation CA→ES and IT→ES

In [ ]:
TRANSLATION_SYSTEM = """You are a professional translator specialised in Romance languages.
Your only task is to translate the text into Spanish.

STRICT RULES:
- Translate faithfully: do not add, remove or modify any content.
- PRESERVE ALL LINE BREAKS (\n) exactly as they appear in the source.
- Do not add comments, explanations or additional text.
- Reply ONLY with the translated text."""


def translate_to_spanish(text: str, source_language: str) -> str:
    prompt = f'Translate the following text from {source_language} to Spanish:\n\n{text}'
    return call_llm(TRANSLATION_SYSTEM, prompt)


def translate_dataframe(df_source, language, id_prefix):
    """Translate a full DataFrame (original_text + simplified_text) into Spanish."""
    rows = []
    for i, row in df_source.iterrows():
        print(f'  seg {row["original_sentence_id"]} ({i+1}/{len(df_source)})...')
        orig_es = translate_to_spanish(row['original_text'],   language); time.sleep(DELAY)
        simp_es = translate_to_spanish(row['simplified_text'], language); time.sleep(DELAY)
        rows.append({
            'document_id':          id_prefix + str(row['document_id']).split('_')[-1],
            'original_sentence_id': row['original_sentence_id'],
            'original_text':        orig_es,
            'simplified_text':      simp_es,
        })
    return pd.DataFrame(rows)

print('Translation functions defined.')

In [ ]:
print(' Translation CAT → ES ')
df_cat_es = translate_dataframe(df_cat, 'Catalan',  'CAT2ES_')
print(f'✓ {len(df_cat_es)} segments')

print()
print(' Translation IT → ES ')
df_it_es = translate_dataframe(df_it, 'Italian', 'IT2ES_')
print(f'✓ {len(df_it_es)} segments')

In [ ]:
df_base = pd.concat([df_es, df_cat_es, df_it_es], ignore_index=True)
df_base.to_csv('dataset_base_es.csv', index=False)

print(f'Base dataset: {len(df_base)} segments')
print(df_base['document_id'].value_counts().to_string())
print('\n✓ Saved to dataset_base_es.csv')

## 5. Back-translation with different language chains per round

### Generic translation prompts

A parameterisable prompt that works for any language pair

In [ ]:
def translate_step(text: str, source_language: str, target_language: str) -> str:
    system = f"""You are a professional translator.
Translate the text from {source_language} to {target_language}.

STRICT RULES:
- Translate faithfully: do not add, remove or modify any content.
- PRESERVE ALL LINE BREAKS (\\n) exactly as they appear in the source.
- Do not add any comments, explanations or extra text.
- Reply ONLY with the translated text."""

    user = f"Translate the following text from {source_language} to {target_language}:\n\n{text}"
    return call_llm(system, user)


def back_translate_chain(text_es: str, chain: list) -> str:
    current_text = text_es
    for source, target in chain:
        current_text = translate_step(current_text, source, target)
        time.sleep(DELAY)
    return current_text


ROUNDS = {
    'bt_round1': {
        'name':   'ES → EN → DE → EN → ES',
        'chain': [
            ('Spanish', 'English'),
            ('English', 'German'),
            ('German',  'English'),
            ('English', 'Spanish'),
        ],
        'reason': 'German has SOV word order, very different from Spanish → greater syntactic variation'
    },
    'bt_round2': {
        'name':   'ES → FR → PT → ES',
        'chain': [
            ('Spanish',    'French'),
            ('French',     'Portuguese'),
            ('Portuguese', 'Spanish'),
        ],
        'reason': 'Romance languages with different vocabulary → moderate but natural lexical variation'
    },
    'bt_round3': {
        'name':   'ES → EN → ZH → EN → ES',
        'chain': [
            ('Spanish', 'English'),
            ('English', 'Chinese'),
            ('Chinese', 'English'),
            ('English', 'Spanish'),
        ],
        'reason': 'Chinese has no inflectional morphology → forces complete sentence restructuring'
    },
}

print('Back-translation chains defined:')
for key, info in ROUNDS.items():
    print(f'  {key}: {info["name"]}')
    print(f'    → {info["reason"]}')

### Full back-translation: 3 rounds over the base dataset

In [ ]:
df_augmented = df_base.copy()
format_warnings = []

for round_name, info in ROUNDS.items():
    print(f'\n{"="*65}')
    print(f'ROUND: {info["name"]}')
    print(f'  Reason: {info["reason"]}')
    print(f'{"="*65}')

    new_rows = []

    for i, row in df_base.iterrows():
        print(f'  [{i+1:02d}/{len(df_base)}] {row["document_id"]} · seg {row["original_sentence_id"]}', end=' ... ')

        bt_orig = back_translate_chain(str(row['original_text']),   info['chain'])
        bt_simp = back_translate_chain(str(row['simplified_text']), info['chain'])

        n_expected = str(row['simplified_text']).count('\n')
        n_obtained  = bt_simp.count('\n')
        if n_expected != n_obtained:
            warning = f'{round_name} | {row["document_id"]} seg{row["original_sentence_id"]}: expected={n_expected} obtained={n_obtained}'
            format_warnings.append(warning)
            print(f'⚠ \\n: {n_expected}→{n_obtained}')
        else:
            print('✓')

        new_rows.append({
            'document_id':          f"{row['document_id']}_{round_name}",
            'original_sentence_id': row['original_sentence_id'],
            'original_text':        bt_orig,
            'simplified_text':      bt_simp,
        })

    df_round     = pd.DataFrame(new_rows)
    df_augmented = pd.concat([df_augmented, df_round], ignore_index=True)
    print(f'  → Round completed. Accumulated total: {len(df_augmented)} segments')

print(f'\n{"="*65}')
print(f'Back-translation completed. Total: {len(df_augmented)} segments')
if format_warnings:
    print(f'\n⚠  {len(format_warnings)} segments with line break discrepancies:')
    for w in format_warnings:
        print(f'   - {w}')
else:
    print(' Line break format (\\n) preserved in all segments.')

### Save and clean the augmented dataset

In [ ]:
before = len(df_augmented)
df_augmented = df_augmented.drop_duplicates(subset=['original_text', 'simplified_text'])
df_augmented = df_augmented[
    df_augmented['original_text'].apply(lambda x: str(x).strip() != '') &
    df_augmented['simplified_text'].apply(lambda x: str(x).strip() != '')
].reset_index(drop=True)
after = len(df_augmented)

print(f'Removed {before - after} duplicates/empty rows.')
print(f'Final dataset: {after} segments')

def classify_origin(doc_id):
    if 'bt_round1' in doc_id: return 'BT: ES→EN→DE→EN→ES'
    if 'bt_round2' in doc_id: return 'BT: ES→FR→PT→ES'
    if 'bt_round3' in doc_id: return 'BT: ES→EN→ZH→EN→ES'
    if 'CAT2ES'    in doc_id: return 'CA→ES'
    if 'IT2ES'     in doc_id: return 'IT→ES'
    return 'ES original'

df_augmented['origin'] = df_augmented['document_id'].apply(classify_origin)

print('\nDistribution by origin:')
print(df_augmented['origin'].value_counts().to_string())

df_augmented.to_csv('es_augmented_dataset.csv', index=False)
print('\n Saved to es_augmented_dataset.csv')

## 6. Token distribution of the augmented dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f'Augmented ES dataset ({len(df_augmented)} seg.) – Token distribution',
    fontsize=13, fontweight='bold'
)

for ax, (col, label, color) in zip(axes, [
    ('orig_tokens', 'Original',   '#4C72B0'),
    ('simp_tokens', 'Simplified', '#DD8452'),
]):
    ax.hist(df_augmented[col], bins=20, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(df_augmented[col].mean(),   color='red',    linestyle='--',
               label=f'Mean={df_augmented[col].mean():.1f}')
    ax.axvline(df_augmented[col].median(), color='orange', linestyle='--',
               label=f'Median={df_augmented[col].median():.1f}')
    ax.set_xlabel('No. tokens')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{label} text')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('dist_tokens_augmented_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dist_tokens_augmented_dataset.png')